# ✏️ Challenge 3: Handwritten Digit AI

This challenge is closer to real image recognition.

Your AI sees a tiny 3×3 image made of 9 pixels.

Pixel numbering:

```text
1 2 3
4 5 6
7 8 9
```

A pixel is either:

- `1` = on
- `0` = off

Your goal is to build an AI that recognises simple digits.


In [ ]:
import numpy as np

pixel_names = [str(i) for i in range(1, 10)]
output_names = ["Zero", "One", "Two", "Seven"]

def make_image(active_pixels):
    pixels = np.zeros(9)
    for p in active_pixels:
        pixels[p - 1] = 1
    return pixels

def show_image(pixels):
    symbols = ["█" if x == 1 else "░" for x in pixels]
    print(symbols[0], symbols[1], symbols[2])
    print(symbols[3], symbols[4], symbols[5])
    print(symbols[6], symbols[7], symbols[8])

def run_digit_ai(pixels, weights, thresholds, show=True):
    pixels = np.array(pixels)
    output_values = pixels @ weights

    active_outputs = []
    for i, value in enumerate(output_values):
        if value >= thresholds[i]:
            active_outputs.append(output_names[i])

    if show:
        print("Input image:")
        show_image(pixels)
        print("\nOutput values:")
        for name, value, threshold in zip(output_names, output_values, thresholds):
            bar = "█" * max(0, int(value))
            print(f"{name:7s}: {value:5.2f} {bar}  threshold={threshold}")
        print("\nPrediction:")
        if len(active_outputs) == 0:
            print("No digit recognised.")
        elif len(active_outputs) == 1:
            print(active_outputs[0])
        else:
            print("Confused:", active_outputs)

    return active_outputs, output_values

def predict_digit(pixels, weights, thresholds):
    active_outputs, output_values = run_digit_ai(pixels, weights, thresholds, show=False)
    if len(active_outputs) == 1:
        return active_outputs[0]
    elif len(active_outputs) == 0:
        return "No decision"
    else:
        return "Confused"


## Part 1 — Draw a digit

Change the active pixels and run the cell.

Examples:

```python
make_image([1,2,3,4,6,7,8,9])  # rough zero
make_image([2,5,8])            # one
make_image([1,2,3,5,7,8,9])    # two
make_image([1,2,3,6,9])        # seven
```


In [ ]:
pixels = make_image([1, 2, 3, 4, 6, 7, 8, 9])
show_image(pixels)


## Part 2 — A first attempt

This AI is not perfect.

Run it and look at the output scores.


In [ ]:
weights = np.array([
    [2, 0, 2, 2],  # pixel 1 -> [Zero, One, Two, Seven]
    [2, 2, 2, 2],  # pixel 2
    [2, 0, 2, 2],  # pixel 3
    [2, 0, 0, 0],  # pixel 4
    [0, 2, 2, 0],  # pixel 5
    [2, 0, 0, 2],  # pixel 6
    [2, 0, 2, 0],  # pixel 7
    [2, 2, 2, 0],  # pixel 8
    [2, 0, 2, 2],  # pixel 9
])

thresholds = np.array([12, 5, 10, 6])

pixels = make_image([1, 2, 3, 4, 6, 7, 8, 9])
run_digit_ai(pixels, weights, thresholds)


## Part 3 — Test the AI

The AI will be tested on several versions of each digit.

Your goal: edit the weights and thresholds to improve the score.

This is a tiny version of training an image classifier.


In [ ]:
test_images = [
    (make_image([1,2,3,4,6,7,8,9]), "Zero"),
    (make_image([1,2,3,4,6,7,9]), "Zero"),
    (make_image([1,2,3,4,6,8,9]), "Zero"),

    (make_image([2,5,8]), "One"),
    (make_image([2,5,8,9]), "One"),
    (make_image([2,4,5,8]), "One"),

    (make_image([1,2,3,5,7,8,9]), "Two"),
    (make_image([1,2,3,5,7,8]), "Two"),
    (make_image([1,2,3,5,8,9]), "Two"),

    (make_image([1,2,3,6,9]), "Seven"),
    (make_image([1,2,3,5,6,9]), "Seven"),
    (make_image([1,2,3,6,8,9]), "Seven"),
]

def test_digit_ai(weights, thresholds, show_mistakes=True):
    correct = 0

    for pixels, answer in test_images:
        prediction = predict_digit(pixels, weights, thresholds)
        if prediction == answer:
            correct += 1
            mark = "✅"
        else:
            mark = "❌"

        print(f"{mark} predicted: {prediction:12s} | correct: {answer}")
        if show_mistakes and prediction != answer:
            show_image(pixels)
            print()

    print(f"Score: {correct}/{len(test_images)}")

test_digit_ai(weights, thresholds)


## Part 4 — Your mission

Edit the weights and thresholds below.

Tips:

- A digit should receive positive weights for pixels it usually uses.
- A digit can receive negative weights for pixels that should usually be off.
- Thresholds decide how much evidence is needed before a neuron fires.
- If many neurons fire at once, make thresholds higher or use negative weights.


In [ ]:
# EDIT THESE NUMBERS

weights = np.array([
    [2, 0, 2, 2],  # pixel 1 -> [Zero, One, Two, Seven]
    [2, 2, 2, 2],  # pixel 2
    [2, 0, 2, 2],  # pixel 3
    [2, 0, 0, 0],  # pixel 4
    [0, 2, 2, 0],  # pixel 5
    [2, 0, 0, 2],  # pixel 6
    [2, 0, 2, 0],  # pixel 7
    [2, 2, 2, 0],  # pixel 8
    [2, 0, 2, 2],  # pixel 9
])

thresholds = np.array([12, 5, 10, 6])

test_digit_ai(weights, thresholds)


## Part 5 — Hard mode: use the highest score

Real classifiers often choose the class with the highest score, even if no threshold is crossed.

Compare the two approaches:

1. Threshold classifier: neurons either fire or do not fire.
2. Highest-score classifier: choose the output with the biggest value.

Which one works better here?


In [ ]:
def predict_digit_highest_score(pixels, weights):
    output_values = np.array(pixels) @ weights
    return output_names[np.argmax(output_values)]

def test_highest_score(weights):
    correct = 0
    for pixels, answer in test_images:
        prediction = predict_digit_highest_score(pixels, weights)
        mark = "✅" if prediction == answer else "❌"
        if prediction == answer:
            correct += 1
        print(f"{mark} predicted: {prediction:7s} | correct: {answer}")
    print(f"Score: {correct}/{len(test_images)}")

test_highest_score(weights)


## Part 6 — Final challenge

Add a fifth output neuron for another digit or symbol.

Ideas:

- Three
- Four
- Eight
- Letter L
- Letter T

You will need to edit:

1. `output_names`
2. `weights`
3. `thresholds`
4. `test_images`

This is how a small classifier becomes a bigger classifier.


In [ ]:
# Optional extension area
